# GridPulse — Week 08 Power BI Export & Gold Hand-off

**Project:** GridPulse: Campus Energy Command Center  
**Team:** Team 05 — GridPulse  
**Week:** 08  
**Purpose:** Prepare approved Gold outputs for Power BI and document the reporting hand-off.

This notebook documents the Week 08 Power BI hand-off. Power BI is intended to consume approved Gold outputs only. The notebook records the Gold sources used for the dashboard, their reporting purpose, export expectations, and validation checks needed before dashboard use.

**Week 09 boundary:** Week 09 refines the same Power BI report and communicates insights; it does not require a new dashboard model.

## 1. Gold Source Register

| Gold source | Reporting purpose | Key fields used | Power BI use |
|---|---|---|---|
| `agg_building_consumption_daily` | Daily building energy consumption and demand | `date_key`, `building_id` | Campus Energy Overview; Building & Load Analysis |
| `agg_peak_load_interval` | Aligned campus peak demand by interval | `date_key`, `interval_ts` | Campus Energy Overview; Building & Load Analysis |
| `agg_energy_cost_daily` | Daily estimated energy cost | `date_key`, `tariff_plan_id` | Campus Energy Overview; Building & Load Analysis |
| `agg_campus_anomaly_daily` | Daily campus anomaly readings and anomaly rate | `date_key` | Meter Health & Live Operations |
| `agg_meter_health_daily` | Daily meter reading counts and power factor | `meter_id`, `building_id`, `date_key` | Meter Health & Live Operations |
| `dim_building` | Building reference information | `building_id` | Building analysis and filtering |
| `dim_date` | Date reference and calendar attributes | `date_key` | Date filtering and time analysis |
| `dim_meter` | Meter reference information | `meter_id`, `building_id` | Meter health and building analysis |
| `dim_tariff` | Tariff reference information | `tariff_plan_id` | Tariff analysis |
| `dim_time_band` | Time-band reference information | time-band fields | Time-band analysis |
| `fact_meter_reading` | Trusted meter-level reading data | `meter_id`, reading timestamp fields | Approved Gold hand-off source where required |

Only approved Gold outputs are intended for Power BI. Bronze, Silver, candidate and quarantine data are not dashboard sources.

## 2. Week 08 Reporting Model

The Power BI report is organized around business questions rather than one visual per Gold table.

### Page 1 — Campus Energy Overview

- Total Energy Consumption
- Total Energy Cost
- Peak Campus Demand
- Average Demand
- Total Buildings
- Daily Campus Energy Consumption
- Energy Consumption by Building
- Daily Peak Campus Demand
- Campus Demand Over Time
- Date and building filters

### Supporting Week 08 model relationships

- `dim_date[date_key]` → `agg_building_consumption_daily[date_key]`
- `dim_date[date_key]` → `agg_peak_load_interval[date_key]`
- `dim_date[date_key]` → `agg_energy_cost_daily[date_key]`
- `dim_building[building_id]` → `agg_building_consumption_daily[building_id]`

Relationships are intended to use one-to-many cardinality with controlled single-direction filtering where applicable. Independent Gold summaries are not directly joined merely because they share a column.

## 3. Power BI KPI Definitions

The Week 08 Power BI measures used for the first dashboard page are:

In [ ]:
# Power BI DAX — documented here for traceability

Total Energy (kWh) =
SUM(agg_building_consumption_daily[total_energy_kwh])

Average Demand (kW) =
AVERAGE(agg_building_consumption_daily[avg_active_power_kw])

Peak Campus Demand =
MAX(agg_peak_load_interval[campus_demand_kw])

Total Energy Cost =
SUM(agg_energy_cost_daily[estimated_energy_cost])

Total Buildings =
DISTINCTCOUNT(agg_building_consumption_daily[building_id])

## 4. Export / Hand-off Checks

The following checks are the Week 08 hand-off requirements. Run them against the approved Gold environment when Databricks access is available.

- Confirm each selected Gold source exists.
- Record source table name, grain and row count.
- Confirm required reporting fields are present.
- Confirm exported field types are suitable for Power BI.
- Preserve the declared Gold grain; do not flatten independent summaries merely for convenience.
- Read back each controlled export and compare rows, columns and selected measures with the owning Gold source.
- Use the same filter state when reconciling important Power BI values to Gold.

**Important:** No row counts or reconciliation results are fabricated in this notebook. Values should be recorded from the actual approved Gold environment or controlled exports.

In [ ]:
# Optional local validation helper for exported CSV files.
# Update EXPORT_DIR if the approved Gold exports are available locally.

from pathlib import Path
import pandas as pd

EXPORT_DIR = Path("../data_sample/gold_exports")

expected_files = [
    "agg_building_consumption_daily.csv",
    "agg_peak_load_interval.csv",
    "agg_energy_cost_daily.csv",
    "agg_campus_anomaly_daily.csv",
    "agg_meter_health_daily.csv",
    "dim_building.csv",
    "dim_date.csv",
    "dim_meter.csv",
    "dim_tariff.csv",
    "dim_time_band.csv",
    "fact_meter_reading.csv",
]

available = []
missing = []

for filename in expected_files:
    path = EXPORT_DIR / filename
    if path.exists():
        available.append(filename)
    else:
        missing.append(filename)

print("Available Gold exports:", len(available))
print("Missing Gold exports:", len(missing))
if missing:
    print("Missing:", missing)

## 5. Export Profile

When the CSV hand-off is available locally, the following cell profiles row counts and columns without changing the source values.

In [ ]:
profiles = []

for filename in expected_files:
    path = EXPORT_DIR / filename
    if path.exists():
        df = pd.read_csv(path)
        profiles.append({
            "file": filename,
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": ", ".join(df.columns.astype(str))
        })

profile_df = pd.DataFrame(profiles)
profile_df

## 6. Power BI Hand-off Checklist

- [x] Approved Gold-only reporting boundary documented.
- [x] Gold source register documented.
- [x] Week 08 Page 1 purpose and visual mapping documented.
- [x] Power BI KPI measures documented.
- [x] Safe relationship pattern documented.
- [ ] Actual export row counts recorded from the final hand-off.
- [ ] Export read-back reconciliation recorded where the source environment is available.
- [ ] Final Power BI reconciliation evidence attached to the Week 08 screenshots/log.

The unchecked items are intentionally left open if the corresponding source/evidence is not available. They should not be replaced with invented values.

## 7. Week 09 Boundary

Week 09 continues from this Week 08 Power BI foundation. The same approved Gold model and Power BI report are refined for page usability, slicer interactions, visual hierarchy, reconciliation evidence and evidence-backed insights.